# 语义内核工具使用示例


## 导入所需的包


In [1]:
import json
import os

from dotenv import load_dotenv

from IPython.display import display, HTML

from typing import Annotated
from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## 创建插件    
Semantic Kernel 使用插件作为代理可以调用的工具。一个插件可以包含多个 `kernel_functions`，作为一个组。

在下面的示例中，我们创建了一个 `DestinationsPlugin`，它包含两个功能：  
1. 使用 `get_destinations` 函数提供目的地列表  
2. 使用 `get_availabilty` 函数提供每个目的地的可用性列表  


In [ ]:
# Define a sample plugin for the sample
class DestinationsPlugin:
    """A List of Destinations for vacation."""

    # 使用 @kernel_function 装饰器，将这个方法暴露给 AI 模型作为工具
    # 功能：提供一个度假目的地列表
    @kernel_function(description="Provides a list of vacation destinations.")
    def get_destinations(self) -> Annotated[str, "Returns the vacation destinations."]:
        return """
        Barcelona, Spain
        Paris, France
        Berlin, Germany
        Tokyo, Japan
        New York, USA
        """

    # 功能：检查某个目的地的可用性
    @kernel_function(description="Provides the availability of a destination.")
    def get_availability(
        self, destination: Annotated[str, "The destination to check availability for."]
    ) -> Annotated[str, "Returns the availability of the destination."]:
        return """
        Barcelona - Unavailable
        Paris - Available
        Berlin - Available
        Tokyo - Unavailable
        New York - Available
        """

## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`ai_model_id` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场中其他可用的模型，以查看不同的结果。

为了使用 `Azure Inference SDK`（用于 GitHub Models 的 `base_url`），我们将在 Semantic Kernel 中使用 `OpenAIChatCompletion` 连接器。此外，还有其他 [可用连接器](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion)，可以使用 Semantic Kernel 连接其他模型提供商。


In [20]:
# 某些环境下需要显式指定行为
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

# 既然你正在研究 Agent 的工具调用（Tool Calling），这个类就是你的“交警”，它负责指挥 LLM（如 Qwen-Max）该如何使用你喂给它的插件。
# 通常你会用到这三种模式：
# Auto (默认)：模型自己决定：是直接回答用户，还是先调用个工具。
# Required：强迫症模式。模型必须先调用至少一个工具，不许直接废话。
# None：禁言模式。即使有工具也不许用。
# 告诉 Agent：你可以自己看着办，想用工具就用，不用也行
settings = FunctionChoiceBehavior.Required()
# 在调用 invoke 或 invoke_stream 时，确保模型被授权使用工具
# 默认情况下，Agent 应该已经包含了这个逻辑，但如果还是空，可以检查底层 Service 的配置。

In [15]:
load_dotenv()
# 创建 AsyncOpenAI 客户端实例

# 使用通义大模型
model_name="qwen-max"
client = AsyncOpenAI(
    api_key=os.environ.get("DASHSCOPE_API_KEY"), 
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# 使用GPT大模型
# model_name = "gpt-4o-mini"
# # 创建 AsyncOpenAI 客户端实例
# client = AsyncOpenAI(
#     api_key=os.environ["GITHUB_TOKEN"],
#     base_url="https://models.inference.ai.azure.com/"
# )

# Create an AI Service that will be used by the `ChatCompletionAgent`
# 创建 AI 服务对象，作为 Semantic Kernel 与 Qwen-Max 模型通信的桥梁
chat_completion_service = OpenAIChatCompletion(
    ai_model_id=model_name,
    async_client=client,
)


## 创建代理
现在我们将通过设置代理名称和指令来创建代理。

您可以更改这些设置，以观察代理响应的不同之处。


In [16]:
# Create the agent
agent = ChatCompletionAgent(
    service=chat_completion_service,
    name="TravelAgent",
    instructions="Answer questions about the travel destinations and their availability.",
    plugins=[DestinationsPlugin()],
)

## 运行代理

现在我们将运行 AI 代理。在这个代码片段中，我们可以向 `user_input` 添加两条消息，以展示代理如何回应后续问题。

代理应该调用正确的函数来获取可用目的地的列表，并确认某个地点的可用性。

你可以更改 `user_inputs` 来查看代理的响应方式。


#### 业务场景：旅行目的地查询系统
+ 用户可以提问：
    + "有哪些可用的度假目的地？"
    + "巴塞罗那有空位吗？"
    + "有哪些不在欧洲的度假目的地？"
+ Agent 会自动调用工具获取数据并返回结果

In [22]:
# 定义模拟用户连续输入的列表，用于测试 Agent 在多轮对话下的表现
user_inputs = [
    "What destinations are available?",
    "Is Barcelona available?",
    "Are there any vacation destinations available not in Europe?",
]

# 定义主异步函数，处理 Agent 的交互逻辑
async def main():
    # 初始化对话线程对象为 None，它将在后续迭代中存储并传递对话历史（上下文）
    thread: ChatHistoryAgentThread | None = None

    # 遍历每一个用户输入的问题
    for user_input in user_inputs:
        # 构建初始 HTML 字符串，用于在 Notebook 中展示用户的提问样式
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        # 初始化当前轮次的变量：Agent 名称、累加文本响应、函数调用日志、已触发函数追踪
        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []
        function_calls_made = []  

        # 初始化缓冲区变量：用于重建流式传输中分块到达的函数名称和 JSON 参数
        current_function_name = None
        argument_buffer = ""

        # 调用 Agent 的流式接口，传入当前消息和历史线程
        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            # 从响应中更新线程对象，确保下一轮对话能携带当前的记忆
            thread = response.thread
            # 获取当前响应包中的 Agent 名称
            agent_name = response.name
            # 将当前流片段中的内容项转换为列表进行遍历
            content_items = list(response.items)

            for item in content_items:
                # 检查当前片段是否为“函数调用”内容（即 LLM 决定使用工具）
                if isinstance(item, FunctionCallContent):
                    # 如果片段中包含函数名，则记录到当前函数名变量中
                    if item.function_name:
                        current_function_name = item.function_name

                    # 由于参数是流式输出的，将获取到的参数片段累加到缓冲区
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                        
                    # 特殊逻辑处理：针对某些版本中函数调用与结果不同步的情况
                    # 如果检测到函数名和参数缓冲区均不为空，则尝试解析并记录
                    if current_function_name and argument_buffer:
                        formatted_args = argument_buffer.strip()
                        try:
                            # 尝试将累加的字符串解析为 JSON，再重新序列化以规范格式
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            # 解析失败则保持原始字符串格式
                            pass 

                        # 将检测到的函数调用信息存入追踪列表
                        function_calls_made.append({
                            'name': current_function_name,
                            'args': formatted_args
                        })
                
                # 检查当前片段是否为“函数执行结果”（通常由框架在本地运行插件后返回）
                elif isinstance(item, FunctionResultContent):
                    # 在显示结果前，先将之前缓冲中挂起的函数调用信息格式化并存入日志
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass 

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        # 处理完毕后清空缓冲区，准备接收下一个可能的调用
                        current_function_name = None
                        argument_buffer = ""

                    # 将函数运行的实际结果（返回值）添加到日志列表中
                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                
                # 检查当前片段是否为常规的“流式文本内容”
                elif isinstance(item, StreamingTextContent) and item.text:
                    # 将模型生成的文本片段累加到最终回答列表中
                    full_response.append(item.text)

        # 补丁逻辑：如果在流中检测到了函数调用但没有捕捉到结果项（常见于某些流模式）
        # 基于已经得到模型文本响应这一事实，推断函数已被成功调用
        if function_calls_made and not function_calls:
            for func_call in function_calls_made:
                function_calls.append(f"Calling function: {func_call['name']}({func_call['args']})")
            
            # 添加一条提示，说明函数结果已被 Agent 内部处理并用于生成上述回复
            if len(function_calls_made) > 0:
                function_calls.append("\nFunction results were processed successfully (results used to generate the response above)")

        # 如果本轮对话涉及函数调用，则构建一个可折叠的 HTML 组件展示详情
        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>" # 使用 HTML5 details 标签实现点击展开效果
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}" # 将函数调用日志用换行符连接并放入容器
                "</div></details></div>"
            )

        # 构建最终的 Assistant 回答部分 HTML，合并所有流式文本片段
        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        # 调用 IPython 的 display 函数，将构建好的 HTML 实时渲染到 Notebook 页面上
        display(HTML(html_output))

# 启动异步主程序
await main()


---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
